# Interacting quantum walkers on `hyperplanes_diff` (qml-benchmarks)

**What this runs.** An interacting multi-walker quantum-walk network (IMW-QNN) and the same network with the
walker interaction switched off (no cross-feature entanglement), on all 19 `hyperplanes_diff` instances, under
the qml-benchmarks protocol: hyperparameters chosen by 5-fold cross-validation on the training set only,
each test set scored once, 5 initialisation seeds. Results are compared with the suite's published results.

**Pre-registration (fixed 24 Sep 2026, before any test evaluation of this model).**
3 walkers on the cycle $C_8$ (9 qubits); learning rate 0.01; 1500 Adam(W) steps; logistic loss.
Grid: layers $\in\{2,4\}$ × weight decay $\in\{10^{-2},10^{-1}\}$. Arms: `interaction=True` (entangling) and
`interaction=False` (control). No further tuning after test results are seen.

**How to run (Colab).**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. *Runtime → Run all.* Allow Google Drive access when asked: results are saved there after every
   instance, so a disconnect loses at most one instance. Re-running resumes where it stopped.
3. When it finishes, download `imw_results.json` (path printed by the last cell) and send it back.


In [ ]:
# ---- configuration -------------------------------------------------------------
SMOKE = False            # True = tiny end-to-end test (2 instances, 50 steps); keep False for the real run
USE_DRIVE = True         # save results to Google Drive (recommended on Colab)
GRID = {"n_layers": [2, 4], "weight_decay": [1e-2, 1e-1]}
FIXED = dict(n_walkers=3, n_vertices=8, learning_rate=0.01, max_steps=1500)
SEEDS = [0, 1, 2, 3, 4]
INSTANCES = list(range(2, 21))
ARMS = {"entangled": True, "no_interaction": False}
if SMOKE:
    GRID = {"n_layers": [2], "weight_decay": [1e-2]}
    FIXED["max_steps"] = 50
    SEEDS = [0, 1]
    INSTANCES = [2, 3]


In [ ]:
# ---- setup ---------------------------------------------------------------------
import os, sys, subprocess
try:
    import optax  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optax"], check=True)
REPO = os.environ.get("QMLB_REPO", "qml-benchmarks")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/XanaduAI/qml-benchmarks.git", REPO], check=True)
sys.path.insert(0, os.path.join(REPO, "src"))

OUT_DIR = "."
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/imw_hyperplanes"
    except Exception as e:
        print("Drive not available, saving locally:", e)
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS = os.path.join(OUT_DIR, "imw_results.json" if not SMOKE else "imw_results_smoke.json")
import jax
print("JAX devices:", jax.devices(), "| results file:", RESULTS)


In [ ]:
# ---- data: exact regeneration of hyperplanes_diff + check against published SVC ------
import numpy as np, pandas as pd, glob
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
# the suite's generator module, imported directly (avoids the package __init__, which pulls in PennyLane)
import importlib.util
_spec = importlib.util.spec_from_file_location("hyperplanes", os.path.join(REPO, "src/qml_benchmarks/data/hyperplanes.py"))
_hp = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_hp)

def load_all():
    np.random.seed(1)                       # paper/benchmarks/generate_hyperplanes.py
    out = {}
    for k in range(2, 21):
        X, y = _hp.generate_hyperplanes_parity(300, 10, k, 3)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2)
        out[k] = (Xtr, ytr, Xte, yte)
    return out

D = load_all()
RES = os.path.join(REPO, "paper/results/hyperplanes_diff")
ok = 0
for k in range(2, 21):
    hp = pd.read_csv(f"{RES}/SVC/SVC_hyperplanes-10d-from3d-{k}n_GridSearchCV-best-hyperparams.csv").set_index("hyperparameter")["best_value"]
    pub = pd.read_csv(f"{RES}/SVC/SVC_hyperplanes-10d-from3d-{k}n_GridSearchCV-best-hyperparams-results.csv")["test_acc"].mean()
    Xtr, ytr, Xte, yte = D[k]
    mine = (SVC(C=float(hp["C"]), gamma=float(hp["gamma"])).fit(Xtr, ytr).predict(Xte) == yte).mean()
    ok += abs(mine - pub) < 1e-9
print(f"published SVC reproduced exactly on {ok}/19 instances")
assert ok == 19, "data does not match the suite's -- stop"


In [ ]:
# ---- model: interacting multi-walker quantum-walk network --------------------------
import jax, jax.numpy as jnp, optax
from sklearn.base import BaseEstimator, ClassifierMixin
CDT = jnp.complex64   # single precision (fast on GPU); probabilities are well conditioned here

def cycle_spectrum(n):
    k = np.arange(n)
    lam = 2 * np.cos(2 * np.pi * k / n)
    F = np.exp(2j * np.pi * np.outer(np.arange(n), k) / n) / np.sqrt(n)
    return jnp.asarray(lam, jnp.float32), jnp.asarray(F, CDT)

class InteractingWalkersClassifier(BaseEstimator, ClassifierMixin):
    """W distinguishable walkers on the cycle C_n, W*log2(n) qubits, all starting at vertex 0.
    Layer l: independent CTQWs exp(-i t_al(x) A) with t_al(x) = theta_al . x + c_al (walker a),
    then the on-site interaction exp(-i g_l sum_{a<b} delta(pos_a, pos_b)), the only gate coupling
    walkers (entangling them and the feature projections they carry).
    Readout: f(x) = alpha <prod_a O_a> + sum_a beta_a <O_a> + b,  O_a = diag(tanh w_a).
    interaction=False fixes g_l = 0: product state over walkers, no cross-feature entanglement."""
    def __init__(self, n_walkers=3, n_vertices=8, n_layers=2, interaction=True,
                 learning_rate=0.01, max_steps=1500, weight_decay=1e-2, random_state=42):
        self.n_walkers = n_walkers; self.n_vertices = n_vertices; self.n_layers = n_layers
        self.interaction = interaction; self.learning_rate = learning_rate
        self.max_steps = max_steps; self.weight_decay = weight_decay; self.random_state = random_state

    def _build(self):
        W, n, L = self.n_walkers, self.n_vertices, self.n_layers
        lam, F = cycle_spectrum(n); Fd = F.conj().T
        grids = np.meshgrid(*[np.arange(n)] * W, indexing="ij")
        pairs = jnp.asarray(sum((grids[a] == grids[b]).astype(np.float32)
                                for a in range(W) for b in range(a + 1, W)))
        inter = self.interaction

        def apply_walker(psi, U, a):
            psi = jnp.moveaxis(psi, a, 0)
            psi = jnp.tensordot(U, psi, axes=(1, 0))
            return jnp.moveaxis(psi, 0, a)

        def state(p, x):
            psi = jnp.zeros((n,) * W, dtype=CDT).at[(0,) * W].set(1.0)
            t = jnp.einsum("ald,d->al", p["theta"], x) + p["c"]
            for l in range(L):
                for a in range(W):
                    U = F @ (jnp.exp(-1j * t[a, l] * lam)[:, None].astype(CDT) * Fd)
                    psi = apply_walker(psi, U, a)
                if inter:
                    psi = psi * jnp.exp(-1j * p["g"][l] * pairs).astype(CDT)
            return psi

        def f(p, x):
            prob = jnp.abs(state(p, x)) ** 2
            o = jnp.tanh(p["w"])
            prod = prob
            for a in range(W):
                prod = jnp.tensordot(o[a], prod, axes=(0, 0))
            singles = jnp.stack([prob.sum(axis=tuple(b for b in range(W) if b != a)) @ o[a] for a in range(W)])
            return p["alpha"] * prod + singles @ p["beta"] + p["b"]

        self._f = jax.jit(jax.vmap(f, in_axes=(None, 0)))
        self._state = jax.jit(state)

    def _init(self, d):
        rng = np.random.default_rng(self.random_state)
        W, n, L = self.n_walkers, self.n_vertices, self.n_layers
        f32 = lambda a: jnp.asarray(a, jnp.float32)
        return {"theta": f32(rng.normal(0, 1 / np.sqrt(d), (W, L, d))), "c": f32(rng.uniform(0, 2 * np.pi, (W, L))),
                "g": f32(rng.uniform(0, 2 * np.pi, L)), "w": f32(rng.normal(0, 1.0, (W, n))),
                "alpha": f32(2.0), "beta": f32(rng.normal(0, 0.5, W)), "b": f32(0.0)}

    def fit(self, X, y):
        X = jnp.asarray(np.asarray(X, np.float32))
        self.classes_ = np.unique(y)
        yy = jnp.asarray(np.where(np.asarray(y) == self.classes_[1], 1.0, -1.0), jnp.float32)
        self._build(); p = self._init(X.shape[1])
        opt = optax.adamw(self.learning_rate, weight_decay=self.weight_decay); fv = self._f
        loss = lambda p: jnp.mean(jax.nn.softplus(-yy * fv(p, X)))
        @jax.jit
        def run(p):
            s = opt.init(p)
            def body(c, _):
                p, s = c
                l, g = jax.value_and_grad(loss)(p)
                u, s = opt.update(g, s, p)
                return (optax.apply_updates(p, u), s), l
            (p, s), ls = jax.lax.scan(body, (p, s), None, length=self.max_steps)
            return p, ls[-1]
        self.params_, l = run(p); self.final_loss_ = float(l)
        return self

    def decision_function(self, X):
        return np.asarray(self._f(self.params_, jnp.asarray(np.asarray(X, np.float32))))
    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, self.classes_[1], self.classes_[0])

    def walker_entanglement(self, X):
        """Entanglement entropy (bits) between walker 0 and the rest, per input."""
        n = self.n_vertices; out = []
        for x in np.asarray(X, np.float32):
            psi = np.asarray(self._state(self.params_, jnp.asarray(x)), np.complex128).reshape(n, -1)
            s = np.linalg.svd(psi, compute_uv=False) ** 2; s = s[s > 1e-12]
            out.append(float(-(s * np.log2(s)).sum()))
        return np.array(out)


In [ ]:
# ---- protocol: GridSearchCV (5-fold, train only) -> 5 seeds -> test once; resumable ----
import json, time, warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import GridSearchCV, StratifiedKFold
cv = StratifiedKFold(5, shuffle=True, random_state=42)
results = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
for k in INSTANCES:
    for arm, inter in ARMS.items():
        key = f"{arm}|{k}"
        if key in results:
            continue
        t0 = time.time()
        Xtr, ytr, Xte, yte = D[k]
        gs = GridSearchCV(InteractingWalkersClassifier(interaction=inter, random_state=42, **FIXED),
                          GRID, cv=cv, n_jobs=1).fit(Xtr, ytr)
        rows = []
        for s in SEEDS:
            m = InteractingWalkersClassifier(interaction=inter, random_state=s, **FIXED, **gs.best_params_).fit(Xtr, ytr)
            rows.append({"seed": s, "train_acc": float((m.predict(Xtr) == ytr).mean()),
                         "test_acc": float((m.predict(Xte) == yte).mean())})
        ent = m.walker_entanglement(Xte)
        results[key] = {"arm": arm, "k": k, "best_params": gs.best_params_, "cv_acc": float(gs.best_score_),
                        "rows": rows, "entanglement_bits_mean": float(ent.mean()), "seconds": time.time() - t0}
        json.dump(results, open(RESULTS, "w"), indent=1)
        te = np.mean([r["test_acc"] for r in rows])
        print(f"k={k:2d} {arm:15s} best={gs.best_params_} cv={gs.best_score_:.3f} test={te:.3f} "
              f"ent={ent.mean():.2f} bits ({time.time()-t0:.0f}s)", flush=True)
print("done:", len(results), "entries in", RESULTS)


In [ ]:
# ---- analysis: against the published leaderboard, and entangled vs control ------------
from scipy import stats
pub = []
for f in glob.glob(f"{RES}/*/*best-hyperparams-results.csv"):
    mname = f.split("/")[-2]; kk = int(os.path.basename(f).split("from3d-")[1].split("n_")[0])
    pub.append((kk, mname, pd.read_csv(f)["test_acc"].mean()))
T = pd.DataFrame(pub, columns=["k", "model", "acc"]).pivot(index="k", columns="model", values="acc")
for arm in ARMS:
    s = {v["k"]: np.mean([r["test_acc"] for r in v["rows"]]) for v in results.values() if v["arm"] == arm}
    if s: T[f"IMW ({arm})"] = pd.Series(s)
T = T.loc[sorted(set().union(*[{v['k'] for v in results.values()}]))]
print(f"Mean test accuracy over {len(T)} instances:")
print(T.mean().sort_values(ascending=False).round(3).to_string())
ours = "IMW (entangled)"
if ours in T and len(T) > 2:
    print("\nPaired comparisons of", ours)
    for m in ["MLPClassifier", "DressedQuantumCircuitClassifier", "DataReuploadingClassifier",
              "IQPKernelClassifier", "IMW (no_interaction)"]:
        if m in T:
            d = T[ours] - T[m]
            print(f"  vs {m:32s} diff {100*d.mean():+.1f} pts  won/lost {(d>1e-9).sum()}/{(d<-1e-9).sum()}  "
                  f"p={stats.ttest_rel(T[ours], T[m]).pvalue:.3f}")
print("\nResults file:", RESULTS)
